# Transcript to Linear

Turn a meeting transcript into a reviewed PRD and Linear tickets. On a later
run, select one linked requirement, revise it using human feedback, and
preview an in-place update to its issue.

Adapted from the [Mistral Cookbook notebook](https://github.com/mistralai/cookbook/blob/main/mistral/agents/non_framework/transcript_linearticket_agent/TranscriptToLinearTicketAgent.ipynb)
and [workflow article](https://mistral.ai/news/agentic-workflows-from-meetings-to-dev-tickets/).

## Run in Colab

1. Add three Colab secrets with notebook access enabled: `MISTRAL_API_KEY`,
   `LINEAR_API_KEY`, and `LINEAR_TEAM_ID`. For the full flow, the Linear
   key needs read, issue creation, and issue update access to the target team.
2. Run the setup cells through **Open project** and grant access to your
   Google Drive. The project JSON is private working state and stores the
   transcript, approved PRD, issue links, and any interrupted apply plan.
3. Run **Review and apply**. On the first run, `APPROVE` previews a new
   batch. `APPLY` publishes it. Use a test team for the sample transcript.
4. Run **Review and apply** again with the same project name to revise a
   linked requirement. Select it by number and enter feedback. After PRD
   approval, inspect the proposed Linear description before typing `APPLY`.

Use one Colab session per project at a time. A project created with this
version starts with an empty link file, so first publish creates new
issues. Tickets from earlier notebook versions have no saved links.


## Installation

In [ ]:
%pip install mistralai==1.5.1 # MistralAI
%pip install "gql[requests]==3.5.0" # GraphQL
%pip install pydantic==2.14.2 # Data validation
%pip install pypdf==5.3.0  # PDF processing

## Imports and configuration

In [ ]:
import copy
import json
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Optional
from difflib import unified_diff
from uuid import uuid4

import requests
from gql import Client, gql
from gql.transport.exceptions import TransportError
from gql.transport.requests import RequestsHTTPTransport
from google.colab import drive, userdata
from IPython.display import Markdown, display
from mistralai import Mistral
from pydantic import BaseModel


In [ ]:
@dataclass
class Config:
    LINEAR_API_KEY: str
    LINEAR_TEAM_ID: str
    LINEAR_GRAPHQL_URL: str
    MISTRAL_API_KEY: str
    MISTRAL_MODEL: str


# Change this value for a different transcript or independent project.
PROJECT_NAME = "le-chat-planning"
config = Config(
    LINEAR_API_KEY=userdata.get("LINEAR_API_KEY"),
    LINEAR_TEAM_ID=userdata.get("LINEAR_TEAM_ID"),
    LINEAR_GRAPHQL_URL="https://api.linear.app/graphql",
    MISTRAL_API_KEY=userdata.get("MISTRAL_API_KEY"),
    MISTRAL_MODEL="mistral-large-latest",
)


## Download the sample transcript

In [ ]:
!wget 'https://raw.githubusercontent.com/mistralai/cookbook/main/mistral/agents/non_framework/transcript_linearticket_agent/lechat_product_call_trascript.pdf' -O './lechat_product_call_trascript.pdf'

## PRD generation and automated review

In [ ]:
def complete_text(api_key, model, prompt, **options):
    """Read text from either a plain or chunked Mistral response."""
    response = requests.post(
        "https://api.mistral.ai/v1/chat/completions",
        headers={"Authorization": f"Bearer {api_key}"},
        json={
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.1,
            **options,
        },
        timeout=(10, 600),
    )
    response.raise_for_status()
    choice = response.json()["choices"][0]
    if choice.get("finish_reason") == "length":
        raise ValueError("Mistral's response was cut off.")
    content = choice["message"]["content"]
    if isinstance(content, list):
        content = "\n".join(block["text"] for block in content if block["type"] == "text")
    if not isinstance(content, str) or not content.strip():
        raise ValueError("Mistral returned no text.")
    return content.strip()


class PRDAgent:
    """Generate and refine a product requirements document from a transcript."""

    def __init__(
        self,
        transcript: str,
        api_key: str,
        model: str = "mistral-large-latest",
    ):
        self.transcript = transcript
        self.api_key = api_key
        self.model = model
        self.prd: Optional[str] = None
        self.feedback: Optional[str] = None

    def _complete_text(self, prompt: str) -> str:
        """Call Mistral and extract the answer text."""
        return complete_text(self.api_key, self.model, prompt)

    def generate_initial_prd(self) -> str:
        """Draft a PRD using the meeting transcript."""
        prompt = f"""
Draft a Product Requirements Document in Markdown. Include the purpose,
scope, features, user personas, technical requirements, constraints,
success metrics, and milestones where the transcript supports them.

Keep reported problems, proposed ideas, and agreed work distinct.
Mark missing targets and unresolved decisions as open questions.

Use a separate `### [REQ-001] Short title` heading for each actionable
implementation requirement, using unique sequential identifiers.
Keep the whole requirement under that heading. Use other headings for
context, proposed ideas, and open product decisions. Only use a REQ
heading for work the transcript supports drafting as implementation.
If there is no such work, use no REQ headings.

Meeting transcript:
{self.transcript}
"""
        self.prd = self._complete_text(prompt)
        self.feedback = None
        return self.prd

    def get_feedback(self) -> str:
        """Review the current PRD against the transcript."""
        if not self.prd:
            raise ValueError("Load or generate a PRD before requesting feedback.")

        prompt = f"""
Review the draft PRD against the meeting transcript. Flag missing
requirements, contradictions, proposals presented as commitments,
and unsupported targets or deadlines. For each finding, identify the
affected PRD section and cite the relevant transcript evidence.

Return feedback only. If no changes are needed, reply exactly: None.

Meeting transcript:
{self.transcript}

Current PRD:
{self.prd}
"""
        self.feedback = self._complete_text(prompt)
        return self.feedback

    def refine_prd(self) -> str:
        """Revise the current PRD using the supplied feedback."""
        if not self.prd or not self.feedback or not self.feedback.strip():
            raise ValueError("A PRD and feedback are required before refinement.")

        prompt = f"""
Revise the PRD using the feedback below. Preserve verified details and
unrelated sections. Keep transcript facts distinct from any new product
decisions explicitly supplied in the feedback. Leave unresolved details
as open questions. Return the complete revised PRD in Markdown. Preserve the ID of each
retained REQ section, and give any new actionable requirement a unique ID.

Meeting transcript:
{self.transcript}

Current PRD:
{self.prd}

Feedback:
{self.feedback}
"""
        self.prd = self._complete_text(prompt)
        return self.prd

    def run(self, max_iterations: int = 3) -> str:
        """Draft the PRD and run up to three automated review rounds."""
        print("Generating initial PRD...", flush=True)
        prd = self.generate_initial_prd()

        for round_number in range(1, max_iterations + 1):
            print(f"\nAutomated review {round_number}/{max_iterations}...", flush=True)
            feedback = self.get_feedback()

            if feedback.strip().lower().rstrip(".") == "none":
                print("No further automated findings.")
                break

            print(feedback)
            prd = self.refine_prd()

        return prd

## Ticket drafting and Linear operations

In [ ]:
class TicketDraft(BaseModel):
    requirement_id: str
    title: str
    description: str


class TicketDrafts(BaseModel):
    tickets: list[TicketDraft]


def read_requirements(prd):
    """Find actionable sections and retain the exact body boundaries."""
    headings = list(re.finditer(r"^#{1,3}[ \t]+([^\n]+)", prd, re.MULTILINE))
    requirements = {}
    for index, heading in enumerate(headings):
        match = re.fullmatch(r"### +\[(REQ-[0-9]{3,})\] +(.+)", heading.group())
        if not match:
            continue
        key, title = match.groups()
        end = headings[index + 1].start() if index + 1 < len(headings) else len(prd)
        body = prd[heading.end():end].strip()
        if key in requirements or not body:
            raise ValueError(f"Each actionable requirement needs a unique ID and body: {key}")
        requirements[key] = {"title": title, "body": body, "start": heading.end(), "end": end}
    return requirements


class TicketCreationAgent:
    def __init__(self, config):
        self.config = config
        self.client = Client(
            transport=RequestsHTTPTransport(
                url=config.LINEAR_GRAPHQL_URL,
                headers={"Authorization": config.LINEAR_API_KEY},
                timeout=30,
            ),
        )

    def parse_prd(self, requirements):
        if not requirements:
            return []
        source = "\n\n".join(
            f"[{key}] {value['title']}\n{value['body']}"
            for key, value in requirements.items()
        )
        prompt = """
Draft exactly one engineering ticket per supplied approved requirement.
Copy its requirement_id exactly. Include a short summary and up to three
testable acceptance criteria. Use numeric targets, standards, SLAs and
implementation choices only when the requirement explicitly sets them.
Keep unresolved details in Open Questions. Return JSON in this form:
{"tickets": [{"requirement_id": "REQ-001", "title": "...", "description": "..."}]}
"""
        answer = complete_text(
            self.config.MISTRAL_API_KEY, self.config.MISTRAL_MODEL,
            prompt + "\n\nApproved requirements:\n" + source,
            response_format={"type": "json_object"}, reasoning_effort="low",
            max_tokens=8192,
        )
        drafts = TicketDrafts.model_validate_json(answer).tickets
        ids = [draft.requirement_id for draft in drafts]
        if len(ids) != len(requirements) or set(ids) != set(requirements):
            raise ValueError("Mistral must return one ticket for each supplied requirement ID.")
        tickets = []
        for draft in drafts:
            ticket = draft.model_dump()
            ticket["title"] = ticket["title"].strip()
            ticket["description"] = ticket["description"].replace(r"\n", "\n").strip()
            if not ticket["title"] or not ticket["description"]:
                raise ValueError(f"Empty ticket fields for {draft.requirement_id}.")
            tickets.append(ticket)
        return tickets

    def get_ticket(self, issue_id):
        issue = self.client.execute(gql("""
            query Issue($id: String!) {
                issue(id: $id) { id identifier title description url team { id } }
            }
        """), variable_values={"id": issue_id})["issue"]
        if issue is None:
            raise ValueError("The linked Linear issue could not be found.")
        if issue["team"]["id"] != self.config.LINEAR_TEAM_ID:
            raise ValueError("The linked issue belongs to a different Linear team.")
        return issue

    def create_ticket(self, issue_id, title, description):
        try:
            result = self.client.execute(gql("""
                mutation CreateIssue($id: String!, $title: String!, $description: String!,
                                     $team: String!) {
                    issueCreate(input: {
                        id: $id, title: $title, description: $description, teamId: $team
                    }) { success issue { id identifier url } }
                }
            """), variable_values={
                "id": issue_id, "title": title, "description": description,
                "team": self.config.LINEAR_TEAM_ID,
            })["issueCreate"]
            issue = result["issue"]
            if not result["success"] or not issue or issue["id"] != issue_id:
                raise RuntimeError("Linear did not confirm the expected issue ID.")
            return issue, True
        except (TransportError, RuntimeError):
            # A timeout can follow a successful write. Resolve using the same saved UUID.
            issue = self.get_ticket(issue_id)
            if issue["title"] != title or (issue.get("description") or "").strip() != description:
                raise RuntimeError("The saved issue ID exists with different content. Review it in Linear.")
            return issue, False

    def update_ticket(self, issue_id, description, expected_description):
        issue = self.get_ticket(issue_id)
        current = issue.get("description") or ""
        if current.strip() == description:
            return False
        if current != expected_description:
            raise ValueError("The Linear description changed after preview. Review before trying again.")
        result = self.client.execute(gql("""
            mutation UpdateIssue($id: String!, $description: String!) {
                issueUpdate(id: $id, input: {description: $description}) {
                    success issue { id }
                }
            }
        """), variable_values={"id": issue_id, "description": description})["issueUpdate"]
        if not result["success"] or (result["issue"] or {}).get("id") != issue_id:
            raise RuntimeError("Linear did not confirm the update. Resume the saved plan to retry.")
        return True


## Workflow orchestration

In [ ]:
class WorkflowOrchestrator:
    """Review the PRD, preview Linear changes, and apply confirmed plans."""

    def __init__(self, config, transcript, state_path):
        self.transcript = transcript
        self.state_path = Path(state_path)
        self.team_id = config.LINEAR_TEAM_ID
        self.prd_agent = PRDAgent(transcript, config.MISTRAL_API_KEY, model=config.MISTRAL_MODEL)
        self.linear_agent = TicketCreationAgent(config)
        self.state = self.load_state() or {
            "team_id": self.team_id, "transcript": transcript,
            "prd": "", "links": {}, "pending": None,
        }
        if self.state["team_id"] != self.team_id:
            raise ValueError("This project is linked to a different Linear team.")

    def load_state(self):
        if self.state_path.exists():
            return json.loads(self.state_path.read_text(encoding="utf-8"))
        return None

    def save_state(self, state):
        self.state_path.parent.mkdir(parents=True, exist_ok=True)
        temporary = self.state_path.with_suffix(".tmp")
        temporary.write_text(json.dumps(state, indent=2), encoding="utf-8")
        temporary.replace(self.state_path)
        self.state = state

    def review_prd(self, prd, revise):
        while True:
            display(Markdown(prd))
            decision = input("APPROVE, REVISE, or STOP: ").strip().upper()
            if decision == "APPROVE":
                return prd
            if decision == "STOP":
                return None
            if decision == "REVISE":
                feedback = input("What should change? ").strip()
                if feedback:
                    prd = revise(prd, feedback)
                    continue
            print("Enter APPROVE, REVISE with feedback, or STOP.")

    def revise_initial(self, prd, feedback):
        self.prd_agent.prd = prd
        self.prd_agent.feedback = feedback
        return self.prd_agent.refine_prd()

    def revise_linked(self, prd, requirement_id, feedback):
        requirements = read_requirements(prd)
        section = requirements[requirement_id]
        # Limit the model's edit to one body. Python preserves all IDs and headings.
        prompt = f"""
Revise only the requirement body below using the reviewer feedback.
Return only the revised body as Markdown, without Markdown headings or
code fences. Preserve requirements and acceptance conditions unrelated
to the feedback. Keep unagreed thresholds, designs and scope as open
questions. Keep human decisions distinct from transcript facts.

Transcript:
{self.transcript}

Requirement: {section["title"]}
Body:
{section["body"]}

Reviewer feedback:
{feedback}
"""
        body = self.prd_agent._complete_text(prompt)
        if re.search(r"^\s*(?:#{1,6}\s|\x60{3})", body, re.MULTILINE):
            raise ValueError("The revision must contain only the requirement body.")
        revised = prd[:section["start"]] + "\n" + body.strip() + "\n\n" + prd[section["end"]:]
        read_requirements(revised)
        return revised

    def make_plan(self):
        if not self.state["prd"]:
            draft = self.prd_agent.run(max_iterations=3)
            prd = self.review_prd(draft, self.revise_initial)
            if prd is None:
                return None
            requirements = read_requirements(prd)
            tickets = self.linear_agent.parse_prd(requirements)
            actions = [
                dict(ticket, kind="create", issue_id=str(uuid4()))
                for ticket in tickets
            ]
            return {"prd": prd, "actions": actions}

        previous = self.state["prd"]
        requirements = read_requirements(previous)
        linked = [key for key in requirements if key in self.state["links"]]
        if not linked:
            print("There are no linked requirements to revise.")
            return None
        for index, key in enumerate(linked, start=1):
            print(f"{index}. [{key}] {requirements[key]['title']}")

        selection = input("Select a requirement number, or press Enter to stop: ").strip()
        if not selection:
            return None
        if not selection.isdecimal() or not 1 <= int(selection) <= len(linked):
            raise ValueError("Select one of the displayed requirement numbers.")
        key = linked[int(selection) - 1]
        feedback = input("Enter feedback for the selected requirement: ").strip()
        if not feedback:
            return None

        def revise(prd, notes):
            return self.revise_linked(prd, key, notes)

        prd = self.review_prd(revise(previous, feedback), revise)
        if prd is None:
            return None
        revised = read_requirements(prd)[key]
        if revised["body"] == requirements[key]["body"]:
            print("No changes to apply.")
            return None

        issue = self.linear_agent.get_ticket(self.state["links"][key])
        draft, = self.linear_agent.parse_prd({key: revised})
        action = dict(
            draft, kind="update", issue_id=issue["id"],
            before=issue.get("description") or "", identifier=issue["identifier"],
            title=issue["title"],
        )
        return {"prd": prd, "actions": [action]}

    def preview(self, plan):
        if not plan["actions"]:
            print("No actionable requirements to publish. APPLY saves the approved PRD.")
        for action in plan["actions"]:
            title = action["title"]
            key = action["requirement_id"]
            if action["kind"] == "create":
                display(Markdown(f"**Create [{key}] {title}**\n\n{action['description']}"))
            else:
                print(f"Update [{key}] {action['identifier']}: {title}")
                diff = unified_diff(
                    action["before"].splitlines(), action["description"].splitlines(),
                    fromfile="Current Linear description", tofile="Proposed description",
                    lineterm="",
                )
                print("\n".join(diff) or "No Linear description changes.")

    def apply_changes(self):
        saved = self.load_state()
        if saved is not None:
            self.state = saved
        plan = self.state["pending"]
        if plan is None:
            print("No changes to apply.")
            return {"created": 0, "updated": 0}

        counts = {"created": 0, "updated": 0}
        for action in plan["actions"]:
            if action.get("done"):
                continue
            key = action["requirement_id"]
            if action["kind"] == "create":
                issue, created = self.linear_agent.create_ticket(
                    action["issue_id"], action["title"], action["description"]
                )
                self.state["links"][key] = issue["id"]
                counts["created"] += int(created)
                print(f"{key} → {issue['identifier']} {issue['url']}")
            else:
                updated = self.linear_agent.update_ticket(
                    action["issue_id"], action["description"], action["before"]
                )
                counts["updated"] += int(updated)
                print(f"{action['identifier']}: {'updated' if updated else 'no changes to apply'}")
            action["done"] = True
            self.save_state(self.state)

        self.state["prd"] = plan["prd"]
        self.state["pending"] = None
        self.save_state(self.state)
        print(f"Created: {counts['created']} | Updated: {counts['updated']}")
        return counts

    def run(self):
        original_state = self.load_state()
        if original_state is not None:
            self.state = copy.deepcopy(original_state)
        plan = self.state["pending"]
        if plan is not None:
            print("Resuming the saved plan. Completed actions will be skipped.")
            for action in plan["actions"]:
                if action["kind"] == "update" and not action.get("done"):
                    issue = self.linear_agent.get_ticket(action["issue_id"])
                    action["before"] = issue.get("description") or ""
        else:
            plan = self.make_plan()
        if plan is None:
            return None

        self.preview(plan)
        if input("Type APPLY to confirm these Linear changes: ").strip().upper() != "APPLY":
            print("No Linear writes made in this run.")
            return None

        # Save the exact plan and preassigned issue IDs before any remote write.
        if self.load_state() != original_state:
            raise ValueError("The project changed since preview. Run the workflow again.")
        self.save_state(dict(self.state, pending=plan))
        return self.apply_changes()


## Open project

In [ ]:
def parse_transcript(config, file_path):
    client = Mistral(api_key=config.MISTRAL_API_KEY)
    with open(file_path, "rb") as pdf:
        document = client.files.upload(
            file={"file_name": Path(file_path).name, "content": pdf},
            purpose="ocr",
        )
    signed_url = client.files.get_signed_url(file_id=document.id)
    result = client.ocr.process(
        model="mistral-ocr-latest",
        document={"type": "document_url", "document_url": signed_url.url},
    )
    return "\n".join(page.markdown for page in result.pages)


In [ ]:
drive.mount("/content/drive")
project_path = Path("/content/drive/MyDrive/transcript-to-linear") / f"{PROJECT_NAME}.json"

if project_path.exists():
    transcript = json.loads(project_path.read_text(encoding="utf-8"))["transcript"]
else:
    transcript = parse_transcript(config, "./lechat_product_call_trascript.pdf")

orchestrator = WorkflowOrchestrator(config, transcript, project_path)


## Review and apply

PRD approval opens the preview. APPLY confirms the Linear operation.

In [ ]:
results = orchestrator.run()


## Saved result

Display the latest applied PRD and recorded links.

In [ ]:
state = orchestrator.load_state()
if state and state["prd"]:
    display(Markdown(state["prd"]))
    for requirement_id, issue_id in state["links"].items():
        print(f"{requirement_id}: {issue_id}")
else:
    print("This project has no applied PRD yet.")
